# GloVe Training Procedure

**Companion wiki page:** https://ml-viz.vercel.app/wiki/glove-training

GloVe on a toy corpus: build the co-occurrence matrix, apply the weighting function, optimize the weighted least-squares objective with gradient descent, and check that similar words end up close.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2d3148'
plt.rcParams['grid.color'] = '#2d3148'
np.random.seed(42)

## A toy corpus and its co-occurrence matrix

$X_{ij}$ counts how often word $j$ appears within a symmetric window around word $i$.

In [ ]:
sentences = [
    "ice is cold cold ice water",
    "steam is hot hot steam water",
    "ice water is solid water",
    "steam water is gas water",
    "cold ice solid ice",
    "hot steam gas steam",
] * 5  # repeat to fatten the counts

tokens = [s.split() for s in sentences]
vocab = sorted({w for sent in tokens for w in sent})
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print("vocab:", vocab)

WINDOW = 2
X = np.zeros((V, V))
for sent in tokens:
    for i, w in enumerate(sent):
        for j in range(max(0, i - WINDOW), min(len(sent), i + WINDOW + 1)):
            if i != j:
                X[w2i[w], w2i[sent[j]]] += 1

print("co-occurrence matrix (rows/cols =", vocab, ")")
print(X.astype(int))

## The weighting function

$f(x) = \min(1, (x/x_{max})^{0.75})$ caps the influence of very frequent pairs and zeroes out pairs that never co-occur.

In [ ]:
X_MAX = 20

def f_weight(x):
    return np.minimum(1.0, (x / X_MAX) ** 0.75)

xs = np.linspace(0, 40, 200)
plt.figure(figsize=(7, 3.5))
plt.plot(xs, f_weight(xs), color='#6366f1', lw=2)
plt.xlabel('co-occurrence count $X_{ij}$'); plt.ylabel('weight $f(X_{ij})$')
plt.title('GloVe weighting: caps frequent pairs, ignores zero pairs')
plt.grid(alpha=0.3); plt.show()

## The objective and its gradients

$$\mathcal{L} = \sum_{i,j} f(X_{ij})\left(\mathbf{v}_i^\top \tilde{\mathbf{v}}_j + b_i + \tilde b_j - \log X_{ij}\right)^2$$

Only nonzero $X_{ij}$ terms contribute. We optimize main vectors $V$, context vectors $\tilde V$, and two bias vectors with plain gradient descent.

In [ ]:
D = 8       # embedding dimension
LR = 0.05
EPOCHS = 400

rng = np.random.default_rng(0)
Vm = rng.normal(scale=0.1, size=(V, D))   # main
Vc = rng.normal(scale=0.1, size=(V, D))   # context
bm = np.zeros(V)
bc = np.zeros(V)

nz = np.argwhere(X > 0)
losses = []
for epoch in range(EPOCHS):
    total = 0.0
    for i, j in nz:
        w = f_weight(X[i, j])
        err = Vm[i] @ Vc[j] + bm[i] + bc[j] - np.log(X[i, j])
        total += w * err ** 2
        g = 2 * w * err
        Vm[i] -= LR * g * Vc[j]
        Vc[j] -= LR * g * Vm[i]
        bm[i] -= LR * g
        bc[j] -= LR * g
    losses.append(total)

emb = Vm + Vc   # final embedding = sum of both matrices
plt.figure(figsize=(8, 4))
plt.plot(losses, color='#6366f1')
plt.xlabel('epoch'); plt.ylabel('weighted least-squares loss')
plt.title('GloVe training loss'); plt.grid(alpha=0.3); plt.show()

## Do similar words end up close?

"ice" should be nearer "cold"/"solid" than "hot"/"gas" — and vice versa for "steam".

In [ ]:
def cos(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

def report(word, probes):
    print(f"similarity to '{word}':")
    for p in probes:
        print(f"  {p:6s} {cos(emb[w2i[word]], emb[w2i[p]]):+.3f}")

report("ice", ["cold", "solid", "hot", "gas"])
print()
report("steam", ["hot", "gas", "cold", "solid"])

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise — the log-count target

**Recap:** GloVe pushes each dot product toward $\log X_{ij}$. After training, the *reconstruction* $\mathbf{v}_i^\top \tilde{\mathbf{v}}_j + b_i + \tilde b_j$ should approximate $\log X_{ij}$ for observed pairs.

Compute the mean absolute reconstruction error over all nonzero pairs (using `Vm, Vc, bm, bc, X, nz` already in memory).

In [ ]:
def mean_abs_error():
    # TODO(you): for each (i, j) in nz, compute the model's prediction
    # and compare to log(X[i, j]); return the mean absolute difference.
    ...

mae = mean_abs_error()
mae

In [ ]:
# Run me — passes silently when correct
errs = [abs(Vm[i] @ Vc[j] + bm[i] + bc[j] - np.log(X[i, j])) for i, j in nz]
expected = float(np.mean(errs))
assert mae is not None and not isinstance(mae, type(Ellipsis)), "fill in the TODO first"
assert abs(float(mae) - expected) < 1e-8
assert expected < 0.5, "training should have brought MAE well below 0.5"
print()

<details>
<summary>Solution</summary>

```python
def mean_abs_error():
    errs = [abs(Vm[i] @ Vc[j] + bm[i] + bc[j] - np.log(X[i, j]))
            for i, j in nz]
    return float(np.mean(errs))
```
</details>